# Tutorial 02 NumPy, Pandas, and Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PEESEgroup/SysEn6888/blob/main/Tutorial_02_NumPy_Pandas_and_Visualization/tutorial_02_data.ipynb)

Turn a small, intentionally messy table of synthetic building measurements into a clean dataset, summary tables, and interpretable plots.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** Python variables, lists, functions, and loops (Tutorial 01).

**Resources:** CPU only. The energy example is embedded; later examples automatically download the small Seaborn tips table and Plotly county data over HTTPS. NumPy, Pandas, Matplotlib, Seaborn, and Plotly install in the notebook if needed. Plan 90–120 minutes for reading and exercises.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required.

## Learning goals

Understand array shape and broadcasting; distinguish element-wise from matrix operations; select and clean tabular data; group and join records; choose appropriate plots; and export your results. The analysis is descriptive, not a predictive model.

The expanded examples cover array construction and attributes, indexing and numerical edge cases, Series alignment, DataFrame editing and MultiIndex, Matplotlib styling, Seaborn facets and categorical plots, and a county choropleth. Sections 4–8 retain the complete synthetic energy analysis. Run every cell in order; intentional error demonstrations are caught and explained.

## 1. Environment

This cell installs missing packages inside the current runtime. It keeps the installed versions of NumPy, Pandas, Matplotlib, Seaborn, and Plotly when available. If imports fail after other experiments, start a fresh runtime and run from the top.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'plotly': 'plotly'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Required packages are already installed.")

In [ ]:
import io, json, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import plotly.express as px
from IPython.display import display
%matplotlib inline

OUTPUT_DIR = Path("outputs/tutorial_02")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "matplotlib": matplotlib.__version__, "seaborn": sns.__version__, "plotly": plotly.__version__}
display(pd.Series(versions, name="Versions"))

## 2. Arrays, shape, and random numbers

NumPy arrays usually store one dtype and support vectorized operations. `shape` tells you the axis lengths; `reshape` changes the arrangement without changing the element count. `arange` excludes the stop, whereas `linspace` includes it by default.

`rng.uniform` and `rng.normal` are different distributions. A standard normal sample can be negative and is not restricted to the interval [0, 1]. A seed reproduces a sequence within a compatible environment.

In [ ]:
vector = np.arange(6, dtype=np.float32)
matrix = vector.reshape(2, 3)
print("Vector:", vector, "shape:", vector.shape, "dtype:", vector.dtype)
print("Matrix:\n", matrix)
print("Five evenly spaced points:", np.linspace(0, 1, 5))
print("Zeros:\n", np.zeros((2, 3)), "\nIdentity:\n", np.eye(3))
uniform_draws = rng.uniform(0, 1, 1000)
normal_draws = rng.normal(0, 1, 1000)
assert ((uniform_draws >= 0) & (uniform_draws < 1)).all()
assert matrix.size == vector.size
print("Normal sample mean/std:", normal_draws.mean(), normal_draws.std())

### 2.1 Construct arrays and inspect their attributes

Create arrays from lists or nested lists. `ones`, `zeros`, and identity matrices are common initializations. A single `-1` lets `reshape` infer one dimension; the number of elements must stay fixed. `argmax` and `argmin` give positions, not values. Signed `int32` spans −2³¹ through 2³¹−1; `float64` uses floating-point representation and does not represent every real number exactly.

In [ ]:
list_array = np.array([1, 2, 3])
nested_array = np.array([[1, 2, 3], [4, 5, 6]])
print("From lists:", list_array, "\n", nested_array)
print("arange:", np.arange(5), np.arange(0, 10, 2))
print("ones:", np.ones(3), "\n", np.ones((2, 3)))
print("zeros:", np.zeros(3), "\n", np.zeros((2, 3)))
np.testing.assert_array_equal(np.identity(3), np.eye(3))
attributes = np.array([4, 2, 9, 1, 5, 8, 3, 7, 6], dtype=np.int32)
print("min/max:", attributes.min(), attributes.max(),
      "argmin/argmax:", attributes.argmin(), attributes.argmax())
print("row:", attributes.reshape(1, 9), "column:\n", attributes.reshape(9, 1))
print("inferred reshape:\n", attributes.reshape(3, -1))
print("shape/size/ndim/dtype:", nested_array.shape, nested_array.size, nested_array.ndim, attributes.dtype)
print("float64 conversion:", attributes.astype(np.float64).dtype)
assert attributes[attributes.argmax()] == attributes.max()
assert attributes.reshape(3, -1).shape == (3, 3)

### 2.2 Uniform, normal, and integer sampling

Use a local `Generator` rather than resetting global random state. `random` is the modern equivalent of `np.random.rand`; `standard_normal` corresponds to `randn`; `integers` corresponds to `randint`. Uniform draws lie in [0, 1), normal draws are unbounded, and integer sampling excludes its upper bound. Multiplying a uniform draw by 2 and adding 10 produces Uniform[10,12), **not** a normal distribution with standard deviation 2.

In [ ]:
sampling_rng = np.random.default_rng(2026)
print("Uniform vector:", sampling_rng.random(3))
print("Uniform matrix:\n", sampling_rng.random((3, 3)))
shifted_uniform = 2 * sampling_rng.random(1000) + 10
print("Shifted uniform range:", shifted_uniform.min(), shifted_uniform.max())
print("Standard normal vector:", sampling_rng.standard_normal(3))
print("Standard normal matrix:\n", sampling_rng.standard_normal((3, 3)))
print("Normal(mean=10, std=2):", sampling_rng.normal(10, 2, 5))
print("Integer scalar:", sampling_rng.integers(1, 10))
integer_draws = sampling_rng.integers(1, 10, size=10)
print("Integer batch:", integer_draws)
assert ((shifted_uniform >= 10) & (shifted_uniform < 12)).all()
assert ((integer_draws >= 1) & (integer_draws < 10)).all()

## 3. Indexing, views, broadcasting, and matrix algebra

`matrix[row, column]` selects a value. Basic slices often share memory with their source; `.copy()` isolates changes. Boolean masks select observations matching a condition.

Broadcasting compares trailing dimensions: dimensions must match or one must equal one. A `[3]` vector can be added to each row of a `[2, 3]` matrix. `*` is element-wise multiplication; `@` is matrix multiplication. Use `np.linalg.solve(A, b)` to solve a system instead of explicitly computing an inverse.

In [ ]:
print("First row:", matrix[0], "Last column:", matrix[:, -1])
print("Values > 2:", matrix[matrix > 2])
independent = matrix[:, :2].copy()
independent[0, 0] = -99
assert matrix[0, 0] == 0
offset = np.array([10, 20, 30])
print("Broadcast addition:\n", matrix + offset)
print("Column means:", matrix.mean(axis=0), "Row means:", matrix.mean(axis=1))
A = np.array([[2., 1.], [1., 3.]])
b = np.array([5., 7.])
solution = np.linalg.solve(A, b)
np.testing.assert_allclose(A @ solution, b)
print("Element-wise square:\n", A * A, "\nMatrix square:\n", A @ A)
print("Solution:", solution, "Transpose:\n", A.T)
try:
    matrix + np.ones(2)
except ValueError as error:
    print("Expected incompatible-shape error:", error)

### 3.1 Index, slice, assign, and reorder

Indices start at zero. Slices exclude the stop. Scalar assignment broadcasts across a selected slice. Basic slices can share memory; integer-array (fancy) indexing returns a copy. Both deliberate out-of-bounds examples below are caught so **Run all** continues.

In [ ]:
index_array = np.arange(1, 10)
print("Scalar and slices:", index_array[0], index_array[3], index_array[2:6], index_array[:4], index_array[4:])
assignment_array = index_array.copy()
assignment_array[:4] = 10
print("Slice assignment:", assignment_array)
assignment_array[:] = 11
print("Whole-array assignment:", assignment_array)
index_matrix = np.arange(1, 10).reshape(3, 3)
assert index_matrix[1][2] == index_matrix[1, 2]
print("Submatrix:\n", index_matrix[:2, 1:], "\nRow:", index_matrix[1], "column:", index_matrix[:, 1])
for label, operation in [("1D index 10", lambda: index_array[10]),
                         ("2D column 3", lambda: index_matrix[1][3])]:
    try:
        operation()
    except IndexError as error:
        print("Expected", label, "IndexError:", error)
    else:
        raise AssertionError("An out-of-bounds access should fail")
row_grid = np.zeros((5, 5))
for row in range(len(row_grid)):
    row_grid[row] = row * 10
print("Selected rows:\n", row_grid[[1, 3, 4]], "\nReordered:\n", row_grid[[4, 1, 0]])
print("Boolean mask:", index_array > 5, "selected:", index_array[index_array > 5])
view_demo = index_array[:2]
view_demo[0] = 99
assert index_array[0] == 99
fancy_demo = index_array[[0, 1]]
fancy_demo[0] = -1
assert index_array[0] == 99
print("Basic slice shares memory:", np.shares_memory(view_demo, index_array))

### 3.2 Vectorized arithmetic and numerical edge cases

NumPy applies arithmetic and universal functions element by element. Zero divided by zero gives `NaN`; the real logarithm of zero gives negative infinity. We suppress only these expected warnings locally and explicitly inspect the results. This is not a recommendation to hide unexpected numerical failures.

In [ ]:
arithmetic = np.arange(5, dtype=float)
print("Add/subtract/multiply/power:", arithmetic + arithmetic, arithmetic - 1,
      arithmetic * arithmetic, arithmetic ** 2)
print("sqrt:", np.sqrt(arithmetic), "exp:", np.exp(arithmetic))
with np.errstate(divide="ignore", invalid="ignore"):
    ratios = arithmetic / arithmetic
    logarithms = np.log(arithmetic)
print("x/x:", ratios, "log(x):", logarithms)
print("Finite ratio mask:", np.isfinite(ratios))
assert np.isnan(ratios[0]) and np.isneginf(logarithms[0])
np.testing.assert_allclose(ratios[1:], 1)
u, v = np.array([1., 2., 3.]), np.array([4., 5., 6.])
print("Vector element-wise:", np.multiply(u, v), "dot:", np.dot(u, v), "@:", u @ v)
print("Matrix element-wise:\n", np.multiply(A, A), "\ndot:\n", np.dot(A, A))
np.testing.assert_allclose(np.dot(A, A), A @ A)
print("Scalar multiplication:\n", 5 * A)
inverse = np.linalg.inv(A)
np.testing.assert_allclose(A @ inverse, np.eye(2), atol=1e-12)
np.testing.assert_allclose(inverse @ b, solution)
print("Inverse (for demonstration):\n", inverse)
print("Vector Euclidean norm:", np.linalg.norm(u), "matrix Frobenius norm:", np.linalg.norm(A))
assert np.isclose(np.linalg.norm(u), np.sqrt((u ** 2).sum()))

## 4. A DataFrame is a labeled table

The embedded CSV below is deliberately small and messy: it contains a duplicate, a missing value, inconsistent site labels, and a negative energy value. It represents **invented** daily records. No claim about real buildings follows from these numbers.

`loc` selects by labels or a mask; `iloc` selects by integer position. A Series is one labeled column. Keep units in column names when possible.

In [ ]:
csv_text = """date,site,temperature_c,energy_kwh
2026-01-01, A,5,24
2026-01-02,A,7,22
2026-01-03,A,9,20
2026-01-04,A,11,18
2026-01-05,A,13,16
2026-01-06,A,15,14
2026-01-01,B,5,31
2026-01-02,B,7,29
2026-01-03,B,9,
2026-01-04,B,11,25
2026-01-05,B,13,23
2026-01-06,B,15,-2
2026-01-02,A,7,22
"""
raw = pd.read_csv(io.StringIO(csv_text))
display(raw.head())
display(raw.dtypes)
print("Rows / columns:", raw.shape)
print("First temperature:", raw.iloc[0]["temperature_c"])
display(raw.loc[raw["temperature_c"] >= 11, ["site", "energy_kwh"]])

### 4.1 Series construction and label alignment

A Series pairs values with labels. It can be built from a list, array, or dictionary, and may store Python objects (`dtype=object`). Numeric Series arithmetic aligns **labels**, even when their order differs. The invented weight/volume values below demonstrate alignment only; they are not physical reference data. A label missing from either operand yields a missing result.

In [ ]:
series_labels = ["oil", "water", "air"]
series_values = [98., 541., 800.]
from_list = pd.Series(series_values, index=series_labels, name="weight")
from_array = pd.Series(np.array(series_values), index=series_labels)
from_dict = pd.Series(dict(zip(series_labels, series_values)))
pd.testing.assert_series_equal(from_array, from_dict)
objects = pd.Series([print, {"example": 1}, np.array([1, 2])], dtype=object)
print("Object Series element types:", objects.map(lambda value: type(value).__name__).tolist())
volume = pd.Series([700., .53, .17], index=["air", "water", "oil"], name="volume")
aligned = pd.DataFrame({"weight": from_list, "volume": volume,
                        "ratio": from_list / volume, "product": from_list * volume,
                        "sum": from_list + volume})
display(aligned)
assert np.isclose(aligned.loc["oil", "ratio"], 98 / .17)
missing_alignment = from_list + pd.Series({"oil": 1., "new_label": 2.})
display(missing_alignment)
assert pd.isna(missing_alignment["water"]) and pd.isna(missing_alignment["new_label"])

### 4.2 DataFrame selection, editing, and filtering

Construct a table from a dictionary or a 2D array. A single column selection gives a Series; a list of columns gives a DataFrame. Most operations return a new object: assign the result or use an explicit in-place operation. A cell-wise mask keeps the original shape and inserts missing values; a row mask selects records. Combine Series conditions with parenthesized `&` or `|`, not Python `and` or `or`.

In [ ]:
dictionary_frame = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6]}, index=["x", "y", "z"])
demo_frame = pd.DataFrame(np.random.default_rng(42).standard_normal((3, 3)),
                          index=["x", "y", "z"], columns=["A", "B", "C"])
display(dictionary_frame, demo_frame)
print("Single column type:", type(demo_frame["A"]).__name__,
      "column-list type:", type(demo_frame[["A"]]).__name__)
display(demo_frame[["A", "C"]], demo_frame.loc["y"], demo_frame.iloc[1])
print("Scalar:", demo_frame.loc["x", "B"])
display(demo_frame.loc[["x", "z"], ["A", "C"]])
edit_frame = demo_frame.copy()
edit_frame["sum"] = edit_frame["A"] + edit_frame["B"]
edit_frame["product"] = edit_frame["A"] * edit_frame["B"]
edit_frame["constant"] = 1
without_sum = edit_frame.drop(columns="sum")
assert "sum" in edit_frame and "sum" not in without_sum
edit_frame.drop(columns="constant", inplace=True)
display(edit_frame, edit_frame.drop(index="y"))
display(demo_frame > 0, demo_frame[demo_frame > 0])
display(demo_frame.loc[demo_frame["A"] > 0, ["B", "C"]])
display(demo_frame.loc[(demo_frame["A"] > 0) & (demo_frame["B"] < 0)])
reset_demo = demo_frame.reset_index()
display(reset_demo)
assert demo_frame.index.tolist() == ["x", "y", "z"]
assert reset_demo.index.tolist() == [0, 1, 2]

### 4.3 Hierarchical indexing with MultiIndex

MultiIndex uses more than one label level. `loc` can select a group; `xs` selects a cross-section at a named level. Give levels meaningful names so the selection is readable. These examples use two groups and four numbered observations each.

In [ ]:
multi_index = pd.MultiIndex.from_product([["A", "B"], [1, 2, 3, 4]], names=["Group", "Number"])
multi_frame = pd.DataFrame(np.arange(16).reshape(8, 2), index=multi_index, columns=["value", "other"])
display(multi_frame)
display(multi_frame.loc["A"], multi_frame.xs("A", level="Group"))
display(multi_frame.xs(("A", 1)), multi_frame.xs(1, level="Number"))
assert multi_frame.xs(1, level="Number").index.tolist() == ["A", "B"]
assert multi_frame.loc[("A", 1), "value"] == 0

## 5. Clean transparently

Normalize labels, parse dates, remove exact duplicates, and mark invalid energy as missing. For this descriptive exercise we drop records missing required values and report every removal. Imputation can be useful, but it requires an explicit assumption. For machine learning, fit any imputation or scaling statistics on the training split only.

In [ ]:
clean = raw.copy()
clean["site"] = clean["site"].str.strip().str.upper()
clean["date"] = pd.to_datetime(clean["date"], errors="raise")
duplicates_removed = int(clean.duplicated().sum())
clean = clean.drop_duplicates().copy()
invalid_energy = clean["energy_kwh"] < 0
invalid_count = int(invalid_energy.sum())
clean.loc[invalid_energy, "energy_kwh"] = np.nan
missing_removed = int(clean[["date", "site", "temperature_c", "energy_kwh"]].isna().any(axis=1).sum())
clean = clean.dropna(subset=["date", "site", "temperature_c", "energy_kwh"]).sort_values(["site", "date"]).reset_index(drop=True)
clean["energy_mj"] = clean["energy_kwh"] * 3.6
audit = {"raw_rows": len(raw), "duplicates_removed": duplicates_removed,
         "invalid_energy_marked_missing": invalid_count, "missing_rows_removed": missing_removed,
         "clean_rows": len(clean)}
assert audit == {"raw_rows":13, "duplicates_removed":1, "invalid_energy_marked_missing":1, "missing_rows_removed":2, "clean_rows":10}
assert (clean.energy_kwh >= 0).all() and not clean.duplicated(["date", "site"]).any()
print(audit)
display(clean)

## 6. Aggregate, pivot, and join

A mean and a total answer different questions. Group summaries should include counts, especially when missing data create unequal coverage. Joining site metadata uses a many-to-one check so duplicated metadata cannot silently multiply rows. A pivot aligns dates across sites; gaps remain visible rather than being invented.

In [ ]:
site_summary = clean.groupby("site").agg(days=("energy_kwh", "count"),
    mean_kwh=("energy_kwh", "mean"), total_kwh=("energy_kwh", "sum"))
site_metadata = pd.DataFrame({"site":["A","B"], "area_m2":[100.,150.]})
enriched = clean.merge(site_metadata, on="site", how="left", validate="many_to_one")
assert len(enriched) == len(clean) and enriched.area_m2.notna().all()
enriched["energy_kwh_per_m2"] = enriched.energy_kwh / enriched.area_m2
pivot = clean.pivot(index="date", columns="site", values="energy_kwh")
display(site_summary)
display(pivot)
print("Total recorded energy:", clean.energy_kwh.sum(), "kWh")
assert clean.energy_kwh.sum() == 222.0

## 7. Choose plots that answer a question

Use a line plot for time, a scatter plot for relationships, and a bar chart for group summaries. Always label units. Here the different totals partly reflect missing days, so the bar chart uses **daily means** and labels the number of observations. These hand-constructed data do not establish causal effects of temperature.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for site in pivot.columns:
    axes[0].plot(pivot.index, pivot[site], marker="o", label=site)
axes[0].set(xlabel="Date", ylabel="Daily energy (kWh)", title="Recorded time series")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(title="Site")
sns.scatterplot(data=clean, x="temperature_c", y="energy_kwh", hue="site", ax=axes[1])
axes[1].set(xlabel="Temperature (°C)", ylabel="Daily energy (kWh)", title="Association, not causation")
bars = axes[2].bar(site_summary.index, site_summary.mean_kwh, color=["#2878a8", "#d28c31"])
axes[2].bar_label(bars, labels=[f"n={v}" for v in site_summary.days], padding=3)
axes[2].set(xlabel="Site", ylabel="Mean daily energy (kWh)", title="Unequal observation counts", ylim=(0,35))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "energy_overview.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(normal_draws, bins=30, alpha=0.65, label="Normal(0, 1)")
ax.hist(uniform_draws, bins=20, alpha=0.65, label="Uniform(0, 1)")
ax.set(xlabel="Draw", ylabel="Count", title="Different sampling distributions")
ax.legend()
plt.show()

## 8. Interactive visualization and export

Hover over points to inspect records. The HTML export includes Plotly’s JavaScript so the saved chart can be opened without fetching a separate script. CSV exports preserve the underlying values for reuse. A correlation matrix summarizes association; with tiny constructed data it is only a demonstration.

In [ ]:
interactive = px.scatter(enriched, x="temperature_c", y="energy_kwh", color="site",
    hover_data=["date", "energy_kwh_per_m2"], labels={"temperature_c":"Temperature (°C)", "energy_kwh":"Daily energy (kWh)"},
    title="Synthetic daily energy measurements")
interactive.show()
interactive.write_html(OUTPUT_DIR / "interactive_energy.html", include_plotlyjs=True)
clean.to_csv(OUTPUT_DIR / "clean_energy.csv", index=False)
site_summary.to_csv(OUTPUT_DIR / "site_summary.csv")
display(clean[["temperature_c", "energy_kwh"]].corr())
reloaded = pd.read_csv(OUTPUT_DIR / "clean_energy.csv", parse_dates=["date"])
pd.testing.assert_frame_equal(reloaded, clean, check_dtype=False)
(OUTPUT_DIR / "metrics.json").write_text(json.dumps({"versions":versions, "audit":audit,
    "total_energy_kwh":float(clean.energy_kwh.sum()), "synthetic":True}, indent=2))
print("Energy analysis complete — tables and figures saved to", OUTPUT_DIR)

## 9. Matplotlib: figures, axes, and presentation

`pyplot` provides a stateful interface; the object-oriented interface makes the target axes explicit. First compare a cubic function with its axes reversed. Then use `add_axes` to place a main axes and an inset using normalized figure coordinates. Titles, labels, and legends should describe the actual plotted quantities.

In [ ]:
plot_x = np.linspace(-20, 20, 101)
plot_y = plot_x ** 3
plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1)
plt.plot(plot_x, plot_y)
plt.xlabel("x"); plt.ylabel("x³"); plt.title("Cubic function")
plt.subplot(1, 2, 2)
plt.plot(plot_y, plot_x, color="tab:orange")
plt.xlabel("x³"); plt.ylabel("x"); plt.title("Axes reversed")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pyplot_subplots.png", dpi=140)
plt.show()
figure = plt.figure(figsize=(9, 5))
main_axes = figure.add_axes([.1, .12, .83, .8])
inset_axes = figure.add_axes([.19, .56, .27, .25])
main_axes.plot(plot_x, plot_x ** 2, label="x²")
main_axes.plot(plot_x, plot_y, label="x³")
main_axes.set(xlabel="x", ylabel="Function value", title="Object-oriented axes with inset")
main_axes.legend(loc="lower right")
inset_x = np.linspace(-1, 1, 40)
inset_axes.plot(inset_x, inset_x ** 2)
inset_axes.set(title="x² near zero")
figure.savefig(OUTPUT_DIR / "axes_inset.png", dpi=140, bbox_inches="tight")
plt.show()

### 9.1 Colors: named, hexadecimal, and survey palettes

Matplotlib accepts named colors, hexadecimal RGB strings, and transparency (`alpha`). CSS4/X11-style names and the XKCD survey can map the **same name** to different RGB values. The following chart compares every shared name using public color dictionaries; the exported table records the exact values. Bold labels indicate identical RGB values.

In [ ]:
from matplotlib.colors import CSS4_COLORS, XKCD_COLORS
from matplotlib.patches import Rectangle
shared_names = sorted(name for name in CSS4_COLORS if "xkcd:" + name in XKCD_COLORS)
color_comparison = pd.DataFrame({"name": shared_names,
    "CSS4": [CSS4_COLORS[name] for name in shared_names],
    "XKCD": [XKCD_COLORS["xkcd:" + name] for name in shared_names]})
color_comparison["same"] = color_comparison.CSS4.str.lower() == color_comparison.XKCD.str.lower()
color_comparison.to_csv(OUTPUT_DIR / "named_color_comparison.csv", index=False)
display(color_comparison.head())
color_columns = 3
color_rows = int(np.ceil(len(color_comparison) / color_columns))
fig, color_axes = plt.subplots(1, color_columns, figsize=(13, color_rows * .32 + 1))
for column, axis in enumerate(color_axes):
    subset = color_comparison.iloc[column * color_rows:(column + 1) * color_rows]
    for row_number, (_, record) in enumerate(subset.iterrows()):
        axis.text(0, row_number + .35, record["name"], fontsize=9,
                  weight="bold" if record["same"] else "normal", va="center")
        for x_position, field in [(1.35, "CSS4"), (2.05, "XKCD")]:
            axis.add_patch(Rectangle((x_position, row_number), .6, .7, color=record[field]))
    axis.set(xlim=(0, 2.75), ylim=(color_rows, -1), title="Name                 CSS4    XKCD")
    axis.axis("off")
fig.suptitle("Shared color names: CSS4 versus XKCD survey", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "named_colors.png", dpi=140, bbox_inches="tight")
plt.show()
print("Compared color names:", len(color_comparison))

### 9.2 Line widths, dashes, and markers

Use line style and markers as well as color to distinguish series. These offset curves demonstrate widths, standard/custom dash patterns, marker shapes and sizes, and separate marker face/edge colors. Offsets are for display only; they do not represent measured differences.

In [ ]:
style_x = np.linspace(0, 2 * np.pi, 12)
fig, style_axes = plt.subplots(1, 3, figsize=(16, 4))
for offset, width in enumerate([.1, .3, .8, 10]):
    style_axes[0].plot(style_x, np.sin(style_x) + offset * 2, linewidth=width, label=f"width={width}")
for offset, (style, color) in enumerate(zip(["-", "-.", ":", "--"], ["red", "tab:blue", "#008866", "purple"])):
    line, = style_axes[1].plot(style_x, np.sin(style_x) + offset * 2, linestyle=style,
                              color=color, alpha=.75, label="custom dash" if offset == 3 else style)
    if offset == 3:
        line.set_dashes([4, 8, 4, 8])
for offset, (marker, size) in enumerate(zip(["s", "1", "+", "o"], [2, 4, 8, 8])):
    style_axes[2].plot(style_x, np.sin(style_x) + offset * 2, marker=marker, markersize=size,
        markerfacecolor="yellow" if offset % 2 else "red", markeredgecolor="green",
        markeredgewidth=3 if offset == 3 else 1, label=f"marker={marker}, size={size}")
for axis, title in zip(style_axes, ["Line width", "Color, alpha, and dashes", "Marker styling"]):
    axis.set(title=title, xlabel="x", ylabel="sin(x) + display offset")
    axis.legend(fontsize=8, loc="upper right")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "line_styles.png", dpi=140, bbox_inches="tight")
plt.show()

### 9.3 Text, rotation, alignment, and boxes

Use axes-relative coordinates (`transAxes`) when text belongs to the layout rather than a data point. Horizontal/vertical alignment determines which part of a text label is anchored. Rotation and box styles can emphasize annotations; avoid hiding the data.

In [ ]:
fig, text_axes = plt.subplots(1, 3, figsize=(10, 3))
for axis, word, rotation, box, color in zip(text_axes, ["We", "Love", "Deep Learning"],
                                         [0, 30, -20], ["round", "square", "circle"],
                                         ["salmon", "orange", "lightcoral"]):
    axis.text(.5, .5, word, transform=axis.transAxes, rotation=rotation,
              ha="center", va="center", fontsize=13,
              bbox={"boxstyle": box, "facecolor": color, "alpha": .65})
    axis.set(title=f"{box}, rotation={rotation}°", xlim=(0, 1), ylim=(0, 1))
    axis.set_xticks([]); axis.set_yticks([])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "text_annotations.png", dpi=140, bbox_inches="tight")
plt.show()

## 10. Seaborn: relational, distribution, and categorical plots

The tips dataset contains 244 restaurant bills and is downloaded automatically from the [Seaborn example-data repository](https://github.com/mwaskom/seaborn-data). Unlike our energy table, these are historical observations. This convenience dataset does not establish causal effects or represent all restaurants. We record download hashes for reproducibility. Internet access is required for these examples and the next section; download failures stop the run rather than substituting data.

In [ ]:
from urllib.request import urlopen
import hashlib

def download_example(url, filename):
    with urlopen(url, timeout=60) as response:
        payload = response.read()
    (OUTPUT_DIR / filename).write_bytes(payload)
    return payload, {"url": url, "sha256": hashlib.sha256(payload).hexdigest(), "bytes": len(payload)}

tips_payload, tips_source = download_example(
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv", "tips.csv")
tips = pd.read_csv(io.BytesIO(tips_payload))
assert len(tips) == 244 and tips[["total_bill", "tip", "size"]].notna().all().all()
display(tips.head())
sns.set_theme(style="whitegrid")
relational = sns.relplot(data=tips, x="total_bill", y="tip", col="time", hue="smoker",
                        style="smoker", size="size", height=3.5)
relational.set_axis_labels("Total bill (USD)", "Tip (USD)")
relational.fig.savefig(OUTPUT_DIR / "tips_relational.png", dpi=140, bbox_inches="tight")
plt.show()
distribution = sns.displot(data=tips, x="total_bill", col="time", kde=True, height=3.5)
distribution.set_axis_labels("Total bill (USD)", "Count")
distribution.fig.savefig(OUTPUT_DIR / "tips_distribution.png", dpi=140, bbox_inches="tight")
plt.show()

### 10.1 Swarm, split violin, and bar charts

A swarm plot shows individual observations. A violin is a smoothed density estimate; its shape depends on sample size and smoothing. A bar chart summarizes the **mean**, not the full distribution. Here bar error bars show bootstrap 95% confidence intervals with a fixed seed. They describe this sample under resampling assumptions and do not imply causal differences. The split violin compares the two smoker categories on each day.

In [ ]:
for kind in ["swarm", "violin", "bar"]:
    options = {"split": True, "inner": "quart"} if kind == "violin" else {}
    if kind == "bar":
        options.update(errorbar=("ci", 95), seed=42, n_boot=1000)
    categorical = sns.catplot(data=tips, x="day", y="total_bill", hue="smoker",
        hue_order=["Yes", "No"], order=["Thur", "Fri", "Sat", "Sun"], kind=kind,
        height=4, aspect=1.5, **options)
    categorical.set_axis_labels("Day", "Total bill (USD)")
    categorical.fig.suptitle(f"Restaurant bills: {kind}", y=1.03)
    categorical.fig.savefig(OUTPUT_DIR / f"tips_{kind}.png", dpi=140, bbox_inches="tight")
    plt.show()

## 11. Plotly: county-level choropleth

Join 2016 unemployment rates to US county polygons using five-character FIPS codes. Preserve leading zeros by reading identifiers as strings. The [Plotly example data](https://github.com/plotly/datasets) include a GeoJSON geometry file and a rate table. This is a historical map, not current unemployment data. Hover to inspect county IDs and rates. A fixed 0–12% color range can saturate values above 12%. The exported HTML embeds Plotly and the county geometry; the geographic basemap may still require an internet connection.

The current files contain 3,219 rate rows, of which 3,217 match the supplied geometry. FIPS `02158` and `46102` are reported as unmatched and omitted from the map; we do not invent boundaries or silently remap identifiers.

In [ ]:
county_payload, county_source = download_example(
    "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json", "counties.geojson")
unemployment_payload, unemployment_source = download_example(
    "https://raw.githubusercontent.com/plotly/datasets/master/fips-unemp-16.csv", "unemployment_2016.csv")
counties = json.loads(county_payload)
unemployment = pd.read_csv(io.BytesIO(unemployment_payload), dtype={"fips": str})
assert unemployment.fips.str.fullmatch(r"\d{5}").all() and unemployment.fips.is_unique
assert unemployment.unemp.notna().all() and (unemployment.unemp >= 0).all()
geometry_ids = {feature["id"] for feature in counties["features"]}
map_data = unemployment.loc[unemployment.fips.isin(geometry_ids)].copy()
unmatched_counties = sorted(set(unemployment.fips) - geometry_ids)
assert len(map_data) > 3000
print("Rate rows:", len(unemployment), "mapped:", len(map_data), "unmatched:", unmatched_counties)
county_map = px.choropleth(map_data, geojson=counties, locations="fips", color="unemp",
    color_continuous_scale="Viridis", range_color=(0, 12), scope="usa",
    labels={"unemp": "Unemployment (%)", "fips": "County FIPS"},
    title="US county unemployment, 2016 (historical example)")
county_map.update_layout(margin={"r": 0, "t": 50, "l": 0, "b": 0}, height=550)
county_map.show()
county_map.write_html(OUTPUT_DIR / "county_unemployment.html", include_plotlyjs=True)
map_data.to_csv(OUTPUT_DIR / "mapped_unemployment.csv", index=False)

## 12. Verify and export the complete run

Check the central results and record the environment, downloaded-data hashes, and output inventory. The completion marker appears only after all examples have run. Download the output folder from Colab before ending your session.

In [ ]:
assert audit["clean_rows"] == 10 and clean.energy_kwh.sum() == 222
assert len(tips) == 244 and len(map_data) > 3000
expected_outputs = ["energy_overview.png", "interactive_energy.html", "clean_energy.csv", "site_summary.csv",
    "pyplot_subplots.png", "axes_inset.png", "named_colors.png", "named_color_comparison.csv",
    "line_styles.png", "text_annotations.png", "tips_relational.png", "tips_distribution.png",
    "tips_swarm.png", "tips_violin.png", "tips_bar.png", "county_unemployment.html", "mapped_unemployment.csv"]
assert all((OUTPUT_DIR / name).stat().st_size > 0 for name in expected_outputs)
metrics = {"versions": versions, "audit": audit, "total_energy_kwh": float(clean.energy_kwh.sum()),
           "energy_data_synthetic": True, "tips_rows": len(tips), "mapped_counties": len(map_data),
           "unmatched_counties": unmatched_counties, "compared_color_names": len(color_comparison),
           "sources": {"tips": tips_source, "counties": county_source, "unemployment": unemployment_source},
           "verified_outputs": expected_outputs}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
print("TUTORIAL_02_COMPLETE — all sections passed; outputs:", OUTPUT_DIR)

## 13. Exercises

1. Predict the output shape of a `[4, 3]` array plus a `[3]` array and test your prediction.
2. Replace listwise deletion with a clearly labeled imputation rule; compare the group means and record the new assumptions.
3. Plot energy per square metre. Does the ranking change? What other differences would matter in a real comparison?
4. Build a Seaborn box plot and explain why ten observations give little information about a distribution’s tails.
5. Read the exported CSV in a fresh cell and reconstruct the daily mean table.

**Concept checks:** `axis=0` reduces across rows; `loc` is label-based; `*` and `@` are different operations; a missing record is not automatically zero; association does not establish causation.

6. Reorder the Series labels and explain why label-aligned division is unchanged.
7. Select Number 2 from both MultiIndex groups and then reset the index.
8. Compare the swarm and violin plots: which display actual observed bills?
9. Change the choropleth color range and explain how it changes the visual emphasis without changing any rates.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_02/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

License information is available in the repository `LICENSE`.

References: [NumPy basics](https://numpy.org/doc/stable/user/basics.html), [Pandas tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/), [Matplotlib](https://matplotlib.org/stable/tutorials/index.html), [Seaborn](https://seaborn.pydata.org/tutorial.html), and [Plotly](https://plotly.com/python/). The building energy measurements are synthetic. The tips and county unemployment examples use the linked historical datasets; download URLs and hashes are recorded in `metrics.json`.